# Myeloid subset integration - pre-processing

## Python modules

In [ ]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

## R modules

In [ ]:
%%R
library(scran)

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(7, 7), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(adata, categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return(adata)

# Fiter out low quality cells (Scnorama, round 1 clustering) 

Based on the round 1 clustering there are cluster in the Scanorama and SCVI embedding that are ribosomal genes low and mitochondrial genes high. That indicates that the cells were damaged or stripped. We just apply a soft filter by removing cells with less than 5% ribosomal reads. 

## Import AnnData 

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_1/myeloid/scanorama.h5ad')

## UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden', 'patient_id', 'cc_phase_class', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=20)

## Quality score distribution

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(20, 5), constrained_layout=True)

fig.suptitle('Quality score distribution', fontsize=16)

sc.pl.violin(adata, ['nCount_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[0, 0], show=False)
sc.pl.violin(adata, ['pMT_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[0, 1], show=False) 
sc.pl.violin(adata, ['nFeature_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[1, 0], show=False) 
sc.pl.violin(adata, ['pRP_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[1, 1], show=False)
sc.pl.scatter(adata, 'pRP_RNA', 'pMT_RNA', color='leiden', legend_loc='on data', size=50, ax=axs[2, 0], show=False)
sc.pl.scatter(adata, 'pRP_RNA', 'nFeature_RNA', color='leiden', legend_loc='on data', size=50, ax=axs[2, 1], show=False)

plt.show()

# Filter cluster 

In [ ]:
adata = adata[adata.obs['leiden'].isin([i for i in adata.obs['leiden'] if i not in ['22', '24']])]

## Filter by ribosomal protein and MT percentage

In [ ]:
adata = adata[(adata.obs['pRP_RNA']>=5) & (adata.obs['pMT_RNA']<=10)]

## Quality score distribution post filter

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(20, 5), constrained_layout=True)

fig.suptitle('Quality score distribution post filter', fontsize=16)

sc.pl.violin(adata, ['nCount_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[0, 0], show=False)
sc.pl.violin(adata, ['pMT_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[0, 1], show=False) 
sc.pl.violin(adata, ['nFeature_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[1, 0], show=False) 
sc.pl.violin(adata, ['pRP_RNA'], groupby='leiden', stripplot=False, inner='box', ax=axs[1, 1], show=False)
sc.pl.scatter(adata, 'pRP_RNA', 'pMT_RNA', color='leiden', legend_loc='on data', size=50, ax=axs[2, 0], show=False)
sc.pl.scatter(adata, 'pRP_RNA', 'nFeature_RNA', color='leiden', legend_loc='on data', size=50, ax=axs[2, 1], show=False)

plt.show()

## Set cell id filter 

In [ ]:
cell_id = adata.obs_names

# Import AnnData and subset by cell_id filter 

In [ ]:
adata = sc.read_h5ad('data/object/pp.h5ad')
adata = adata[adata.obs_names.isin(cell_id)]

## Subset genes

In [ ]:
adata = adata[:, (adata.X>=3).sum(axis=0)>=3]

## Scran normalization factors 

In [ ]:
def input_groups(adata): 
    
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    sc.pp.pca(adata, n_comps=15, svd_solver='arpack')
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata, key_added='groups', resolution=0.5)
    
    print(adata.obs['groups'].value_counts())
    
    # Preprocess variables for scran normalization
    input_groups = adata.obs['groups']
    
    return input_groups

In [ ]:
# Scran data 
input_groups = input_groups(adata.copy())
data_mat = adata.X.T.todense()

In [ ]:
%%R -i data_mat -i input_groups -o size_factors

size_factors <- BiocGenerics::sizeFactors(scran::computeSumFactors(SingleCellExperiment::SingleCellExperiment(list(counts=data_mat)), clusters=input_groups, min.mean=0.1))

In [ ]:
with open('data/object/subsets/round_2/myeloid/size_factors.pkl', 'wb') as f:
    pickle.dump(size_factors, f)
f.close()

In [ ]:
size_factors = pd.read_pickle(r'data/object/subsets/round_2/myeloid/size_factors.pkl')

In [ ]:
adata.obs['size_factors'] = size_factors

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, figsize=(15, 5), constrained_layout=True)

sc.pl.scatter(adata, 'size_factors', 'nCount_RNA', show=False, ax=ax1, size=50)
sc.pl.scatter(adata, 'size_factors', 'nFeature_RNA', show=False, ax=ax2, size=50)
ax3 = sb.distplot(size_factors, bins=50, kde=False)
ax3.set(xlabel='size_factors', ylabel='counts')

plt.show()

## HVG selection 

In [ ]:
def hvg_select(adata, key, key_min_size=150, key_min_n=None, min_disp=0.5, file=None): 
    
    print('############ HVG selection ############')
    
    # Check key size
    key_size = adata.obs[key].value_counts()
    print('Data sets N=' + str(len(key_size)))
    print(key_size)
    
    # Select key by sample size 
    key_size = key_size[key_size>key_min_size]
    key_select = key_size.index.to_list()
    print('Sample size select: '+ str(len(key_size)))
    
    # Set min key size if not int for hvg selection 
    print('Sample size min for HVG selection: ', str(key_min_n))
    
    # Subset by selected keys
    adata = adata[adata.obs[key].isin(key_select)]
    
    # Normalize data 
    adata.X /= adata.obs['size_factors'].values[:,None]
    adata.X = sparse.csr_matrix(adata.X)
    sc.pp.log1p(adata)
    
    # Highly variable genes 
    sc.pp.highly_variable_genes(adata, flavor='cell_ranger', min_disp=min_disp, batch_key=key)
    sc.pl.highly_variable_genes(adata)
    
    hvg = adata.var_names[(adata.var['highly_variable']) & (adata.var['highly_variable_nbatches']>=key_min_n)].to_list()
    print('Number hvg :' + str(len(hvg)))
    
    # Write output
    if file is not None: 
        with open(file, 'wb') as f:
            pickle.dump(hvg, f)
        f.close()
    
    return(hvg)

In [ ]:
hvg_1 = hvg_select(adata.copy(), key='sample_name', key_min_n=1, key_min_size=150, min_disp=0.1, file='data/object/subsets/round_2/myeloid/hvg_1.pkl')

In [ ]:
hvg_2 = hvg_select(adata.copy(), key='sample_name', key_min_n=1, key_min_size=150, min_disp=0.5, file='data/object/subsets/round_2/myeloid/hvg_2.pkl')

# Processing functions 

## Scanorama integration 

In [ ]:
def scanorama_integrate(adata, hvg, key, file=None):
    
    # Normalize data 
    adata.X /= adata.obs['size_factors'].values[:,None]
    adata.X = sparse.csr_matrix(adata.X)
    sc.pp.log1p(adata)
    
    # Scale 
    sc.pp.scale(adata, zero_center=True)    
    
    # Subset by hvg 
    adata = adata[:, adata.var_names.isin(hvg)].copy()
    
    # Subset data by i  
    adata_key = dict()
    for i in adata.obs[key].unique():
        adata_key[i] = adata[adata.obs[key]==i]
        
    adata_key = list(adata_key.values())
    
    # Scanorama with count matrix correction 
    scanorama.integrate_scanpy(adata_key, dimred=100, verbose=False)
    adata = ad.concat(adata_key, join="inner")
    
    # Set colors
    adata = set_color(adata, list(color.keys()))
    
    # Reset full gene matrix 
    adata_raw = adata.raw.to_adata()
    
    adata = adata.raw.to_adata()
    adata.raw = adata_raw
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

## SCVI integration 

In [ ]:
def scvi_integrate(adata, hvg, model_dir=None, file=None): 
    
    # Subset by hvg 
    adata = adata[:, adata.var_names.isin(hvg)].copy()
    
    scvi.model.SCVI.setup_anndata(
        adata,
        batch_key='sample_name'
    )
    
    model = scvi.model.SCVI(
        adata, 
        n_latent=10, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'
    )
    
    model.train()
    
    # Write output 
    if model_dir is not None: 
        model.save(model_dir, overwrite=True)
    
    adata.obsm['X_scvi'] = model.get_latent_representation()
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return adata

## Dim reduction and cluster function 

In [ ]:
def dim_clust(adata, use_rep=None, n_neighbors=15, resolution=1, min_dist=0.5, metric='euclidean', file=None): 
    
    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    sc.tl.leiden(adata, resolution=resolution)
    
    sc.tl.umap(adata, min_dist=min_dist)
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

# Scanorama

In [ ]:
adata_0 = scanorama_integrate(adata.copy(), hvg=adata.var_names, key='sample_name', file='data/object/subsets/round_2/myeloid/scanorama.h5ad')
adata_0 = dim_clust(adata_0, use_rep='X_scanorama', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scanorama.h5ad')

In [ ]:
sc.pl.umap(adata_0, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# Scanorama (HVG dispersion 0.1)

In [ ]:
adata_1 = scanorama_integrate(adata.copy(), hvg=hvg_1, key='sample_name', file='data/object/subsets/round_2/myeloid/scanorama_hvg_1.h5ad')
adata_1 = dim_clust(adata_1, use_rep='X_scanorama', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scanorama_hvg_1.h5ad')

In [ ]:
sc.pl.umap(adata_1, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# Scanorama (HVG dispersion 0.5)

In [ ]:
adata_2 = scanorama_integrate(adata.copy(), hvg=hvg_2, key='sample_name', file='data/object/subsets/round_2/myeloid/scanorama_hvg_2.h5ad')
adata_2 = dim_clust(adata_2, use_rep='X_scanorama', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scanorama_hvg_2.h5ad')

In [ ]:
sc.pl.umap(adata_2, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# SCVI 

In [ ]:
adata_3 = scvi_integrate(adata.copy(), hvg=adata.var_names, model_dir='data/object/subsets/round_2/myeloid/model', file='data/object/subsets/round_2/myeloid/scvi.h5ad')
adata_3 = dim_clust(adata_3, use_rep='X_scvi', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scvi.h5ad')

In [ ]:
sc.pl.umap(adata_3, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# SCVI (HVG dispersion 0.1)

In [ ]:
adata_4 = scvi_integrate(adata.copy(), hvg=hvg_1, model_dir='data/object/subsets/round_2/myeloid/model_hvg_1', file='data/object/subsets/round_2/myeloid/scvi_hvg_1.h5ad')
adata_4 = dim_clust(adata_4, use_rep='X_scvi', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scvi_hvg_1.h5ad')

In [ ]:
sc.pl.umap(adata_4, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)

# SCVI (HVG dispersion 0.5)

In [ ]:
adata_5 = scvi_integrate(adata.copy(), hvg=hvg_2, model_dir='data/object/subsets/round_2/myeloid/model_hvg_2', file='data/object/subsets/round_2/myeloid/scvi_hvg_2.h5ad')
adata_5 = dim_clust(adata_5, use_rep='X_scvi', n_neighbors=15, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/round_2/myeloid/scvi_hvg_2.h5ad')

In [ ]:
sc.pl.umap(adata_5, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'celltypist_immun_high', 'celltypist_immun_low', 'pRP_RNA', 'pMT_RNA', 'nCount_RNA', 'nFeature_RNA'], legend_loc='right margin', ncols=3, wspace=0.3, size=20)